Copyright (c) MONAI Consortium  
Licensed under the Apache License, Version 2.0 (the "License");  
you may not use this file except in compliance with the License.  
You may obtain a copy of the License at  
&nbsp;&nbsp;&nbsp;&nbsp;http://www.apache.org/licenses/LICENSE-2.0  
Unless required by applicable law or agreed to in writing, software  
distributed under the License is distributed on an "AS IS" BASIS,  
WITHOUT WARRANTIES OR CONDITIONS OF ANY KIND, either express or implied.  
See the License for the specific language governing permissions and  
limitations under the License.

# Multimodal Early-Fusion Network: Radiographs + Clinical Tabular Data

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/Project-MONAI/tutorials/blob/main/multimodal/nakaseke_multimodal_early_fusion/multimodal_early_fusion_tutorial.ipynb)

This tutorial builds an end-to-end **early-fusion** classifier that combines a 2D medical image stream with a low-dimensional clinical tabular stream, using only MONAI's dictionary-based transform and dataset APIs plus a small PyTorch fusion network.

The clinical schema (age, BMI, salivary pH, systolic blood pressure) and the binary screening task are modeled after a real hypertension-screening workflow at Nakaseke Hospital, Uganda. Because that patient data is confidential and cannot be published, **this notebook generates a fully synthetic cohort locally** -- no downloads, no external services, no real patient data -- while keeping the same feature schema, tensor shapes, and modeling problem, so the pipeline is a drop-in template for a real (IRB-approved, de-identified) dataset with matching keys.

The synthetic generator ties both modalities to a shared hidden "risk factor" per patient, so neither the image nor the tabular vector is fully predictive on its own -- this is what makes the fusion architecture worth demonstrating, rather than a stream that could win alone.

**Why early fusion, and why it fits this problem specifically.** Fusion strategies are usually grouped into early (feature-level), late (decision-level), and joint/intermediate variants. Late fusion -- training an image classifier and a tabular classifier independently and averaging their predictions -- is the more common shortcut, but it caps the model's expressive power at the sum of two univariate opinions: it cannot learn that, say, a mildly ambiguous radiograph combined with a specific BMI/BP combination is jointly more informative than either reading alone. Early fusion, by concatenating learned embeddings from both streams before a shared classification head, lets the network learn exactly those cross-modal interactions. The cost is exactly what this notebook is designed to show how to pay safely: image and tabular data live on very different scales and shapes, and MONAI's dictionary transforms are what make combining them straightforward without writing a custom `Dataset`.


## Setup environment

In [ ]:
!python -c "import monai" || pip install -q "monai-weekly[nibabel, tqdm]"
!python -c "import matplotlib" || pip install -q matplotlib
%matplotlib inline

## Setup imports

In [ ]:
import os
import shutil
import tempfile
from typing import Any

import matplotlib.pyplot as plt
import nibabel as nib
import numpy as np
import torch
import torch.nn as nn
from monai.config import print_config
from monai.data import DataLoader, Dataset
from monai.metrics import ROCAUCMetric
from monai.networks.nets import DenseNet121
from monai.transforms import Compose, EnsureChannelFirstd, EnsureTyped, LoadImaged, ScaleIntensityRanged
from monai.utils import set_determinism

print_config()

## Reproducibility

MONAI's `set_determinism` seeds Python's `random`, NumPy, and PyTorch (including CUDA) in one call, so the synthetic cohort, model initialization, and training loop below are reproducible run to run.

In [ ]:
set_determinism(seed=42)

## Setup data directory

You can specify a directory with the `MONAI_DATA_DIRECTORY` environment variable.  
This allows you to save results and reuse the generated files.  
If not specified, a temporary directory is used and removed at the end of the notebook.

In [ ]:
directory = os.environ.get("MONAI_DATA_DIRECTORY")
if directory is not None:
    os.makedirs(directory, exist_ok=True)
root_dir = tempfile.mkdtemp() if directory is None else directory
print(root_dir)

## The clinical data generator (Nakaseke-inspired, fully synthetic)

`simulate_nakaseke_multimodal_dataset` creates, entirely on disk locally:
- one 2D synthetic radiograph slice per patient, saved as a standard NIfTI (`.nii.gz`) file with `nibabel`
- a 4-dimensional tabular vector `[age, bmi, salivary_ph, systolic_bp]`
- a binary screening label

and returns a MONAI-style data manifest: a list of dictionaries `{"image": path, "nakaseke_tabular": array, "label": int}`.

**On the radiograph itself:** this is a *stylized synthetic density map*, not a rendering of real anatomy -- generated by low-pass filtering white noise into a smooth spatial field, then applying a radial vignette so density fades toward the edges (see the visualization below). The point is not to look like a diagnostic-quality X-ray; it is to give the CNN stream a spatially *smooth*, *bounded* structure whose average density is deliberately correlated with `risk_factor` -- unlike independent per-pixel noise, which a convolutional network could still exploit through its mean but which would not visually resemble anything a clinician would recognize as tissue.

In [ ]:
def _generate_smooth_density_field(rng: np.random.Generator, image_size: int, cutoff: float = 0.12) -> np.ndarray:
    """Low-pass filter white noise into a smooth, blob-like spatial field.

    Real tissue density varies smoothly in space; independent per-pixel noise
    does not, and looks like static rather than anatomy. An FFT low-pass
    filter is the simplest way to get that smoothness using only NumPy, so
    the notebook stays free of extra plotting/image dependencies.
    """
    noise = rng.normal(0.0, 1.0, size=(image_size, image_size))
    freqs = np.fft.fftfreq(image_size)
    freq_x, freq_y = np.meshgrid(freqs, freqs)
    radial_frequency = np.sqrt(freq_x**2 + freq_y**2)
    low_pass_mask = (radial_frequency < cutoff).astype(np.float32)
    smoothed = np.fft.ifft2(np.fft.fft2(noise) * low_pass_mask).real
    return smoothed / (smoothed.std() + 1e-6)


def simulate_nakaseke_multimodal_dataset(
    root_dir: str,
    num_patients: int = 200,
    image_size: int = 64,
    seed: int = 42,
) -> list[dict[str, Any]]:
    """Generate a synthetic multimodal cohort mimicking the Nakaseke Hospital
    hypertension-screening schema (2D radiograph + 4 clinical features).

    No real patient data is used or required. A shared latent ``risk_factor``
    per patient drives both the systolic blood pressure and the mean
    radiograph density, so that neither modality alone is fully predictive
    of the label.
    """
    image_dir = os.path.join(root_dir, "nakaseke_synthetic_images")
    os.makedirs(image_dir, exist_ok=True)

    rng = np.random.default_rng(seed)
    affine = np.eye(4)
    data_manifest: list[dict[str, Any]] = []

    # Radial vignette: fades density toward the edges so each synthetic scan
    # reads as one bounded structure rather than texture filling the frame.
    # It is identical for every patient, so it is computed once, outside the loop.
    yy, xx = np.mgrid[0:image_size, 0:image_size]
    center = (image_size - 1) / 2
    radius_from_center = np.sqrt((xx - center) ** 2 + (yy - center) ** 2)
    vignette = np.clip(1.0 - (radius_from_center / radius_from_center.max()) ** 1.5, 0.0, 1.0)

    for patient_idx in range(num_patients):
        risk_factor = rng.normal(loc=0.0, scale=1.0)

        age = float(np.clip(rng.normal(45, 15), 18, 90))
        bmi = float(np.clip(rng.normal(24, 5), 15, 45))
        salivary_ph = float(np.clip(rng.normal(6.8, 0.4), 5.5, 8.0))
        systolic_bp = float(np.clip(125 + 15 * risk_factor + rng.normal(0, 8), 90, 200))
        tabular_features = np.array([age, bmi, salivary_ph, systolic_bp], dtype=np.float32)

        base_intensity = 90.0 + 25.0 * risk_factor
        density_field = _generate_smooth_density_field(rng, image_size)
        structure = 55.0 * density_field * vignette
        fine_grain = rng.normal(0, 6, size=(image_size, image_size))
        radiograph = np.clip(base_intensity + structure + fine_grain, 0, 255).astype(np.float32)

        label_logit = risk_factor + rng.normal(0, 0.6)
        label = int(label_logit > 0)

        scan_path = os.path.join(image_dir, f"patient_{patient_idx:04d}.nii.gz")
        nib.save(nib.Nifti1Image(radiograph, affine), scan_path)

        data_manifest.append(
            {
                "image": scan_path,
                "nakaseke_tabular": tabular_features,
                "label": label,
            }
        )

    return data_manifest


data_manifest = simulate_nakaseke_multimodal_dataset(root_dir, num_patients=200, image_size=64)
train_files, val_files = data_manifest[:160], data_manifest[160:]

print(
    f"Generated {len(data_manifest)} synthetic patient records "
    f"({len(train_files)} train / {len(val_files)} validation)."
)
print("Example record keys:", list(data_manifest[0].keys()))
print("Example tabular vector [age, bmi, salivary_ph, systolic_bp]:", data_manifest[0]["nakaseke_tabular"])
print("Example label:", data_manifest[0]["label"])

## Advanced dictionary transforms: preserving the tabular stream

The `Compose` pipeline below only ever targets the `"image"` key for image-specific processing
(`LoadImaged`, `EnsureChannelFirstd`, `ScaleIntensityRanged` to normalize scanner variation that is
common across rural-clinic radiograph equipment). The `"nakaseke_tabular"` and `"label"` entries never
pass through any image transform -- they are only explicitly cast to PyTorch tensors with `EnsureTyped`,
which is what lets a single dictionary-based pipeline carry heterogeneous, non-image data safely
alongside imaging data.

In [ ]:
multimodal_transforms = Compose(
    [
        LoadImaged(keys="image", image_only=True),
        EnsureChannelFirstd(keys="image"),
        ScaleIntensityRanged(keys="image", a_min=0, a_max=255, b_min=0.0, b_max=1.0, clip=True),
        EnsureTyped(keys=["nakaseke_tabular", "label"]),
    ]
)

train_ds = Dataset(data=train_files, transform=multimodal_transforms)
val_ds = Dataset(data=val_files, transform=multimodal_transforms)

train_loader = DataLoader(train_ds, batch_size=8, shuffle=True, num_workers=0)
val_loader = DataLoader(val_ds, batch_size=8, shuffle=False, num_workers=0)

sanity_batch = next(iter(train_loader))
print("Sanity check on one batch:")
print("  image shape:            ", sanity_batch["image"].shape)
print("  nakaseke_tabular shape: ", sanity_batch["nakaseke_tabular"].shape)
print("  label shape:            ", sanity_batch["label"].shape)

### Visualize one synthetic sample

Expect a soft, blobby grayscale pattern that fades out toward the edges -- not a sharp anatomical image, and not flat noise either. If it instead looks like pure salt-and-pepper static, `_generate_smooth_density_field`'s FFT low-pass step did not run; if it looks like a hard geometric grid, the vignette exponent is too aggressive for the chosen `image_size`.

In [ ]:
sample = val_ds[0]
plt.imshow(sample["image"][0], cmap="gray")
plt.title(f"Synthetic radiograph | label={int(sample['label'])}")
plt.axis("off")
plt.show()

## The multimodal early-fusion architecture

`ResilientMultimodalClassifier` has two independent streams and a fusion junction:

- **Stream 1 (visual):** a MONAI `DenseNet121` (`spatial_dims=2, in_channels=1, out_channels=512`) that
  turns each radiograph into a 512-dimensional embedding.
- **Stream 2 (tabular context):** a small feed-forward projection network that compresses the
  4-dimensional Nakaseke tabular vector into a 16-dimensional embedding.
- **Fusion junction:** the two embeddings are concatenated with `torch.cat` into a single
  528-dimensional (512 + 16) fused representation.
- **Classification head:** a dropout-regularized linear head maps the fused representation to the
  final diagnostic logit.

**Design intuition worth calling out explicitly:**

- *Why the 512 vs. 16 embedding asymmetry?* An image carries far more raw entropy than a 4-value
  vector, so collapsing both streams to the same width would either waste capacity on the tabular side
  or force the image into an unnecessarily narrow bottleneck. 16 dimensions is enough for a linear
  projection of 4 clinical features to be useful in the fused vector without dominating it. This is a
  practical default, not a tuned optimum -- a natural extension is a learned gating or attention layer
  that lets the network decide the relative weight of each stream per patient instead of fixing it
  through embedding width alone.
- *Why Dropout in the head, specifically?* Real single-site clinical tabular cohorts are usually small
  (tens to a few hundred patients), while `DenseNet121` alone carries roughly 7-8 million parameters.
  Dropout on the fused representation is a cheap, standard regularizer for exactly this
  small-data/large-model regime; for a production deployment on a real cohort this size, pairing it
  with weight decay and early stopping on a held-out validation loss is advisable.
- *Why not just concatenate the raw 4 tabular values instead of projecting them?* Because the raw
  values live on very different scales (age in years vs. salivary pH in single digits), and
  concatenating unprojected features next to a 512-d embedding would let the image stream dominate the
  fused gradient by sheer dimensionality. A small learned projection gives the optimizer a
  same-order-of-magnitude representation to work with on both sides of `torch.cat`.

In [ ]:
class ResilientMultimodalClassifier(nn.Module):
    """Early-fusion network combining a 2D radiograph stream with a tabular clinical stream."""

    def __init__(
        self,
        tabular_in_features: int = 4,
        tabular_embed_dim: int = 16,
        image_embed_dim: int = 512,
        num_classes: int = 1,
        dropout: float = 0.3,
    ) -> None:
        super().__init__()

        self.image_stream = DenseNet121(
            spatial_dims=2,
            in_channels=1,
            out_channels=image_embed_dim,
        )

        self.tabular_stream = nn.Sequential(
            nn.Linear(tabular_in_features, 32),
            nn.ReLU(),
            nn.Linear(32, tabular_embed_dim),
            nn.ReLU(),
        )

        fused_dim = image_embed_dim + tabular_embed_dim
        self.classification_head = nn.Sequential(
            nn.Dropout(dropout),
            nn.Linear(fused_dim, 64),
            nn.ReLU(),
            nn.Dropout(dropout),
            nn.Linear(64, num_classes),
        )

    def forward(self, image: torch.Tensor, tabular: torch.Tensor) -> torch.Tensor:
        image_embedding = self.image_stream(image)
        tabular_embedding = self.tabular_stream(tabular)
        fused = torch.cat([image_embedding, tabular_embedding], dim=1)
        return self.classification_head(fused)

### Prove the fusion graph with a single mock forward pass

In [ ]:
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
model = ResilientMultimodalClassifier().to(device)

mock_images = torch.randn(4, 1, 64, 64, device=device)
mock_tabular = torch.randn(4, 4, device=device)
mock_logits = model(mock_images, mock_tabular)

print("Mock fused output shape:", mock_logits.shape)
assert mock_logits.shape == (4, 1), "Fusion graph produced an unexpected output shape."
print("Fusion graph OK: image (512-d) + tabular (16-d) -> 528-d fused vector -> 1 logit.")

## Train the fusion model

A short training loop over the synthetic cohort. `max_epochs` and `val_interval` follow the MONAI
tutorial convention so the automated notebook-execution tests can safely reduce them for CI.

In [ ]:
max_epochs = 5
val_interval = 1
learning_rate = 1e-3

loss_function = nn.BCEWithLogitsLoss()
optimizer = torch.optim.Adam(model.parameters(), lr=learning_rate)

for epoch in range(max_epochs):
    model.train()
    epoch_loss = 0.0
    for batch in train_loader:
        images = batch["image"].to(device)
        tabular = batch["nakaseke_tabular"].to(device)
        labels = batch["label"].to(device).float().unsqueeze(1)

        optimizer.zero_grad()
        logits = model(images, tabular)
        loss = loss_function(logits, labels)
        loss.backward()
        optimizer.step()
        epoch_loss += loss.item()

    epoch_loss /= len(train_loader)
    print(f"epoch {epoch + 1}/{max_epochs} average training loss: {epoch_loss:.4f}")

    if (epoch + 1) % val_interval == 0:
        model.eval()
        correct, total = 0, 0
        with torch.no_grad():
            for batch in val_loader:
                images = batch["image"].to(device)
                tabular = batch["nakaseke_tabular"].to(device)
                labels = batch["label"].to(device).float().unsqueeze(1)

                logits = model(images, tabular)
                predictions = (torch.sigmoid(logits) > 0.5).float()
                correct += (predictions == labels).sum().item()
                total += labels.numel()

        print(f"epoch {epoch + 1}/{max_epochs} validation accuracy: {correct / total:.4f}")

### Report a threshold-independent clinical metric

Accuracy depends on the 0.5 decision threshold and is a poor summary for screening tasks with any
class imbalance -- it is easy to look good on accuracy by mostly predicting the majority class. The
ROC-AUC is threshold-independent and is the metric clinical ML work is generally expected to report,
so this notebook computes it once on the full validation set with MONAI's native `ROCAUCMetric`.

In [ ]:
auc_metric = ROCAUCMetric()
model.eval()
with torch.no_grad():
    for batch in val_loader:
        images = batch["image"].to(device)
        tabular = batch["nakaseke_tabular"].to(device)
        labels = batch["label"].to(device).float().unsqueeze(1)

        probabilities = torch.sigmoid(model(images, tabular))
        auc_metric(y_pred=probabilities, y=labels)

final_val_auc = auc_metric.aggregate()
auc_metric.reset()
print(f"Final validation ROC-AUC: {final_val_auc:.4f}")

## Cleanup data directory

Remove the temporary directory if one was used.

In [ ]:
if directory is None:
    shutil.rmtree(root_dir)